# vector_checker — полный цикл в ноутбуке

1. Конфиг (`DF_PATH`, `QUERY_PATH`)
2. **Prepare** — подставить ключи в запрос для 1С (Excel ещё не нужен)
3. Выполнить запрос в 1С → положить Excel в `work/excel/`
4. **Compare** — сверка и разбор расхождений

In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vector_checker.compare_features import (
    compare_feature_frames,
    feature_match_stats,
    load_excel_export,
    mismatched_keys,
    paired_compare_frame,
    report_to_frame,
    side_by_side_for_loss,
)
from vector_checker.paths import (
    DEFAULT_QUERY_OUT,
    DEFAULT_SAMPLE_META,
    EXCEL_DIR,
    KEY_COL,
    REPORTS_DIR,
    ensure_work_dirs,
)
from vector_checker.query_inject import (
    normalize_loss_number,
    sample_loss_numbers,
    unique_loss_numbers,
    write_injected_query,
    write_sample_meta,
)
from vector_checker.synthetic import write_synthetic_fixtures

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
ensure_work_dirs()
print("REPO_ROOT:", REPO_ROOT)

## 1. Конфиг путей

- `USE_SYNTHETIC=True` — локальный smoke без 1С (сразу есть synthetic Excel)
- боевой режим: `USE_SYNTHETIC=False`, укажи `QUERY_PATH` и `DF_PATH`
- Excel на этом шаге **не нужен** — его проверим после prepare

In [ ]:
USE_SYNTHETIC = False
SYNTHETIC_MISMATCH = True

# --- боевой режим ---
QUERY_PATH = REPO_ROOT / "queries" / "Сутяжность.txt"
DF_PATH = Path(r"C:\path\to\df_for_service.parquet")  # замени
EXCEL_NAME = "result.xlsx"  # файл в work/excel/

N_LOSSES: int | None = 50  # None = все ключи
RANDOM_SEED = 42
KEY = KEY_COL  # обычно LOSS_NUMBER
# None = только пересечение колонок с Excel (вектор модели из 1С),
# не все 360 колонок датасета. Или явный список: ["FILIAL", "APPLICANT_AGE", ...]
FEATURES: list[str] | None = None
QUERY_OUT = DEFAULT_QUERY_OUT
SAMPLE_META = DEFAULT_SAMPLE_META

if USE_SYNTHETIC:
    paths = write_synthetic_fixtures()
    df_path = paths["df"]
    excel_path = paths["excel_bad" if SYNTHETIC_MISMATCH else "excel_ok"]
    query_path = paths["query_stub"]
else:
    df_path = Path(DF_PATH)
    query_path = Path(QUERY_PATH)
    excel_path = EXCEL_DIR / EXCEL_NAME
    if not df_path.is_file():
        raise FileNotFoundError(df_path)
    if not query_path.is_file():
        raise FileNotFoundError(query_path)

print("query :", query_path)
print("df    :", df_path)
print("excel :", excel_path, "(проверим после prepare)")
print("sample:", N_LOSSES, "seed:", RANDOM_SEED)
print("excel exists:", excel_path.is_file())

## 2. Prepare — подставить ключи в запрос 1С

Excel на этом шаге не нужен. Результат: `work/query_for_1c.txt`.

In [ ]:
service_full = pd.read_parquet(df_path)
if KEY not in service_full.columns:
    raise KeyError(f"В датасете нет колонки {KEY}: {list(service_full.columns)[:20]}...")

all_losses = unique_loss_numbers(service_full[KEY].tolist())
sampled = sample_loss_numbers(all_losses, N_LOSSES, random_seed=RANDOM_SEED)

write_injected_query(
    query_path,
    sampled,
    QUERY_OUT,
    source_df=df_path,
    as_strings=False,  # int без кавычек — иначе фильтр Убыток.Номер в 1С не работает
    n_total=len(all_losses),
    random_seed=RANDOM_SEED if N_LOSSES is not None else None,
)
write_sample_meta(
    SAMPLE_META,
    sampled,
    source_df=df_path,
    query_path=query_path,
    n_total=len(all_losses),
    n=N_LOSSES,
    random_seed=RANDOM_SEED,
)

print(f"ключей всего : {len(all_losses)}")
print(f"в выборке    : {len(sampled)}" + (f" (n={N_LOSSES}, seed={RANDOM_SEED})" if N_LOSSES is not None else ""))
print(f"запрос для 1С: {QUERY_OUT}")
print(f"sample meta  : {SAMPLE_META}")
print(f"положите Excel в: {EXCEL_DIR} / {EXCEL_NAME}")
display(service_full.head(3))

## 3. Excel из 1С

1. Открой `work/query_for_1c.txt` в консоли запросов 1С
2. Выполни и выгрузи результат в Excel
3. Положи файл как `work/excel/result.xlsx` (или поменяй `EXCEL_NAME` в конфиге)
4. Дальше запускай ячейки сверки

## 4. Загрузка Excel и сверка

Здесь уже нужен Excel. Если файла нет — вернись к шагу 3.

In [ ]:
if not excel_path.is_file():
    raise FileNotFoundError(
        f"Нет Excel: {excel_path}\n"
        f"Сначала выполни prepare, выгрузи результат из 1С и положи файл сюда."
    )

if "service_full" not in dir() or "sampled" not in dir():
    service_full = pd.read_parquet(df_path)
    sampled = sample_loss_numbers(
        service_full[KEY].tolist(), N_LOSSES, random_seed=RANDOM_SEED
    )

excel_raw = load_excel_export(excel_path)
if KEY not in excel_raw.columns:
    raise KeyError(f"В Excel нет колонки {KEY}: {list(excel_raw.columns)[:20]}")

_keys = set(sampled)
service = service_full.loc[
    service_full[KEY].map(normalize_loss_number).isin(_keys)
].copy()
excel = excel_raw.loc[
    excel_raw[KEY].map(normalize_loss_number).isin(_keys)
].copy()

# Для просмотра/сверки — только KEY + колонки вектора (Excel / FEATURES)
if FEATURES:
    view_cols = [KEY] + [c for c in FEATURES if c != KEY]
else:
    view_cols = [KEY] + [c for c in excel.columns if c != KEY]
view_cols = [c for c in view_cols if c in service.columns]
service_view = service[view_cols].copy()

print("excel raw :", excel_raw.shape, "→ after key filter:", excel.shape)
print("service   :", service_view.shape, f"(из {service_full.shape}, sample={len(sampled)})")
print("view cols :", len(view_cols), view_cols[:12], "...")
display(service_view.head(3))
display(excel.head(3))

## 5. Сводка и расхождения

1. совпадение значений по общим фичам (`feat_stats`)
2. фичи из Excel, которых нет в service
3. фичи из service, которых нет в Excel

In [ ]:
report, mismatches = compare_feature_frames(
    service,
    excel,
    features=FEATURES,
    all_overlap=FEATURES is None,
    key_col=KEY,
)

# 2 строки на убыток (service/excel), только расходящиеся колонки — не long-таблица
paired = paired_compare_frame(
    service,
    excel,
    mismatches=mismatches,
    features=FEATURES,
    only_mismatch_cols=True,
    key_col=KEY,
)
paired_path = REPORTS_DIR / "vector_compare_paired.csv"
paired.to_csv(paired_path, index=False, encoding="utf-8-sig")

feat_stats = feature_match_stats(
    mismatches,
    features=report.features_compared,
    n_keys=report.n_matched_keys,
)
feat_stats_path = REPORTS_DIR / "vector_compare_feature_stats.csv"
feat_stats.to_csv(feat_stats_path, index=False, encoding="utf-8-sig")

display(report_to_frame(report))
print("ok:", report.ok)
print("mismatched keys:", len(mismatched_keys(mismatches, key_col=KEY)))
print("paired saved:", paired_path, paired.shape)
print("feature stats:", feat_stats_path)

print("\n1) сверка значений (общие фичи):", len(report.features_compared))
display(feat_stats)
fully_ok = feat_stats.loc[feat_stats["match_pct"] == 100.0, "feature"].tolist() if not feat_stats.empty else []
print("   полностью совпали (100%):", len(fully_ok), fully_ok[:30], ("..." if len(fully_ok) > 30 else ""))

print("\n2) есть в Excel, нет в service:", len(report.features_missing_in_service))
print("  ", report.features_missing_in_service or "—")

print("\n3) есть в service, нет в Excel:", len(report.features_missing_in_excel))
print("  ", report.features_missing_in_excel or "—")

print("\n(полный paired — в csv, вертикальный пример — в §7)")

## 6. Ключи только в одной стороне

In [ ]:
only_service = pd.DataFrame({KEY: report.only_service_losses})
only_excel = pd.DataFrame({KEY: report.only_excel_losses})
print("только в service:", len(only_service))
display(only_service.head(20))
print("только в excel:", len(only_excel))
display(only_excel.head(20))

## 7. Один случайный кейс несовпадения

Если есть расхождения — вертикально (`feature | service | excel`) по одному случайному `LOSS_NUMBER`.

In [ ]:
bad_keys = mismatched_keys(mismatches, key_col=KEY)
if not bad_keys:
    print("Несовпадений по фичам нет — нечего показывать.")
else:
    rng = random.Random(RANDOM_SEED)
    KEY_TO_INSPECT = rng.choice(bad_keys)
    print("случайный кейс:", KEY_TO_INSPECT, f"(из {len(bad_keys)} ключей с расхождениями)")
    sbs = side_by_side_for_loss(
        service, excel, KEY_TO_INSPECT, features=FEATURES, key_col=KEY
    )
    display(sbs.loc[~sbs["match"]].reset_index(drop=True))